# 🔀 Routing with RunnableBranch

The **routing** workflow classifies an input, then sends it down **one**
specialist path. Each category gets its own prompt, and no other path runs.

This notebook rebuilds the creative-content router from
`05_AI_Agent_Fundamentals/4. Workflow_Pattern/2. Routing/` — same three
categories, same three test inputs — using `RunnableBranch` instead of
`add_conditional_edges`.

```
                  ┌──▶ story chain ──┐
user request ──▶ classify ──▶ poem chain ──┼──▶ output
                  └──▶ joke chain ───┘
                       (exactly ONE runs)
```

## Learning Objectives
In this notebook, you will learn:
1. **Structured classification** - force a routing decision into a schema with `with_structured_output`
2. **RunnableBranch** - dispatch to one of several chains based on a condition
3. **The default branch** - why `RunnableBranch` always needs a fallback path
4. **Dict-lookup routing** - the lighter alternative when conditions are simple equality checks
5. **Workflow vs agent routing** - who decides the path, your code or the model

## Prerequisites
- An `OPENAI_API_KEY` in a `.env` file at the repo root (or swap the model below)
- `langchain >= 1.0`, `langchain-core >= 1.0`, `pydantic >= 2`
- Completion of `6.1_Prompt_Chaining.ipynb`

---

## 🔧 1. Setting Up the Environment

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and initialise the chat model
# ============================================================================
import warnings

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableBranch, RunnableLambda, RunnablePassthrough
from pydantic import BaseModel, Field
from typing_extensions import Literal

warnings.filterwarnings("ignore")
load_dotenv()

llm = init_chat_model("openai:gpt-4o-mini", temperature=0)

print(f"✅ Model ready: {llm.__class__.__name__}")

---

## 🏷️ 2. The Routing Schema

A router is only as reliable as its classifier. Free-text answers like
`"I think they want a poem"` are unusable as routing keys, so we constrain the
model's output to a `Literal` of exactly three values.

`with_structured_output` returns a runnable that emits a `ContentRoute` instance
rather than an `AIMessage`, so `.content_type` is guaranteed to be one of the
three strings.

In [ ]:
# ============================================================================
# ROUTING SCHEMA: Constrain the classifier to three valid destinations
# ============================================================================


class ContentRoute(BaseModel):
    """Schema for routing creative content requests."""

    content_type: Literal["story", "poem", "joke"] = Field(
        description="The type of creative content to generate"
    )


classifier = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Analyze the user's request and determine if they want a story, poem, "
            "or joke. Consider keywords, tone, and intent in your decision.",
        ),
        ("human", "{user_input}"),
    ]
) | llm.with_structured_output(ContentRoute)

print("✅ Classifier ready — output is constrained to story / poem / joke")

---

## ✍️ 3. The Three Specialist Chains

Each destination is an ordinary LCEL chain. In the LangGraph version these were
three node functions writing to `final_output`; here they are three runnables,
and only the selected one is ever invoked.

In [ ]:
# ============================================================================
# SPECIALIST CHAINS: One per content type
# ============================================================================

story_chain = (
    ChatPromptTemplate.from_template("Write an engaging short story based on: {user_input}")
    | llm
    | StrOutputParser()
)

poem_chain = (
    ChatPromptTemplate.from_template("Create a creative poem inspired by: {user_input}")
    | llm
    | StrOutputParser()
)

joke_chain = (
    ChatPromptTemplate.from_template("Write a funny, clean joke about: {user_input}")
    | llm
    | StrOutputParser()
)

print("✅ Three specialist chains defined")

---

## 🔀 4. Dispatching with RunnableBranch

`RunnableBranch` takes `(condition, runnable)` pairs and a **mandatory default**
as the last positional argument. It evaluates conditions **in order** and runs
the first runnable whose condition returns true. Exactly one branch runs.

The default is not optional decoration. If every condition fails and there is no
default, `RunnableBranch` raises. The LangGraph version had the same safety net
in `routing_map.get(..., "generate_story")`.

> **Key insight**: the conditions read `routing_decision`, a value the classifier
> already produced. The branch itself makes no LLM call — it is pure Python
> dispatch, which is what makes this a *workflow* rather than an agent.

In [ ]:
# ============================================================================
# ROUTER: Classify first, then branch on the decision
# ============================================================================

# assign() runs the classifier and stores its verdict alongside the original
# input, so the specialist chains still see {user_input}.
content_router = RunnablePassthrough.assign(
    routing_decision=classifier | RunnableLambda(lambda route: route.content_type)
) | RunnableBranch(
    (lambda payload: payload["routing_decision"] == "story", story_chain),
    (lambda payload: payload["routing_decision"] == "poem", poem_chain),
    (lambda payload: payload["routing_decision"] == "joke", joke_chain),
    story_chain,  # default — required, and reached only if the schema is bypassed
)

print("✅ Router assembled")

---

## 🧪 5. Testing the Router

The same three test inputs as the LangGraph notebook. To show the decision as
well as the output, we run the classifier stage separately first, then the full
router.

In [ ]:
# ============================================================================
# TEST INPUTS: One per expected destination
# ============================================================================
test_inputs = [
    "Tell me something funny about artificial intelligence",
    "I want to hear a tale about a brave knight",
    "Create something poetic about the ocean",
]

for user_request in test_inputs:
    print(f"\n--- Processing: '{user_request}' ---")

    decision = classifier.invoke({"user_input": user_request})
    output = content_router.invoke({"user_input": user_request})

    print(f"🏷️  Routed to: {decision.content_type}")
    print(f"📄 Output: {output[:120]}...")

---

## 🪶 6. The Lighter Alternative: Dict Lookup

When every condition is a plain equality check on the same field, three lambdas
are more ceremony than the job needs. A `RunnableLambda` may **return a
runnable**, and LCEL will run it — so a dictionary lookup does the same work.

Both forms are correct. Reach for `RunnableBranch` when conditions are genuinely
different predicates, such as checking input length, a regex, or a confidence
score. Reach for the dict when you are mapping one label to one chain.

In [ ]:
# ============================================================================
# DICT ROUTING: Same dispatch, expressed as a lookup table
# ============================================================================
ROUTES = {
    "story": story_chain,
    "poem": poem_chain,
    "joke": joke_chain,
}


def pick_chain(payload: dict):
    """Return the runnable for this decision; LCEL invokes what we return."""
    return ROUTES.get(payload["routing_decision"], story_chain)


dict_router = RunnablePassthrough.assign(
    routing_decision=classifier | RunnableLambda(lambda route: route.content_type)
) | RunnableLambda(pick_chain)

print("✅ Dict router assembled")
print(dict_router.invoke({"user_input": "Tell me a joke about debugging"})[:200])

---

## ⚖️ 7. Workflow Routing vs Agent Routing

There is a second way to build a router in LangChain 1.x: wrap each specialist
chain as a **tool** and let `create_agent` choose. That is the "subagents"
pattern from the LangChain multi-agent docs.

The two are not interchangeable:

| | `RunnableBranch` (this notebook) | `create_agent` with sub-agent tools |
|---|---|---|
| Who picks the path | your code, from a constrained label | the model, at run time |
| Paths per request | exactly one | zero, one, or several |
| Cost | 1 classify call plus 1 specialist call | unbounded loop until the model stops |
| Debuggability | the decision is a value you can log and assert on | the decision is inside the model's tool calls |
| Category | **workflow** | **agent** |

Pick the branch when your categories are known and stable. Pick the agent when
the request may need several specialists, or none, and you cannot enumerate the
combinations in advance.

In [ ]:
# ============================================================================
# COST CHECK: Confirm the branch runs exactly ONE specialist chain
# ============================================================================
call_log = []


def counting(name: str, chain):
    """Wrap a chain so we can see whether it was invoked."""
    return RunnableLambda(lambda payload: (call_log.append(name), chain.invoke(payload))[1])


counted_router = RunnablePassthrough.assign(
    routing_decision=classifier | RunnableLambda(lambda route: route.content_type)
) | RunnableBranch(
    (lambda p: p["routing_decision"] == "story", counting("story", story_chain)),
    (lambda p: p["routing_decision"] == "poem", counting("poem", poem_chain)),
    (lambda p: p["routing_decision"] == "joke", counting("joke", joke_chain)),
    counting("story", story_chain),
)

counted_router.invoke({"user_input": "Write me a haiku about rain"})
print(f"🔧 Specialist chains invoked: {call_log}")
print(f"🧮 Count: {len(call_log)} (a router runs exactly one)")

---

## 📝 Summary

We rebuilt the LangGraph content router in LCEL with `RunnableBranch`.

### 1. The translation table

| LangGraph | LCEL |
|---|---|
| `add_conditional_edges(node, fn, map)` | `RunnableBranch((cond, chain), ..., default)` |
| the `routing_map.get(..., default)` fallback | the mandatory last argument to `RunnableBranch` |
| three handler nodes writing `final_output` | three chains, only one invoked |
| `with_structured_output(ContentRoute)` | identical — this API is unchanged in 1.x |

### 2. What to remember
- **RunnableBranch runs exactly one branch.** First matching condition wins, and
  the rest are never evaluated. That is what separates routing from
  parallelization, which runs them all.
- **The default is mandatory.** It is your protection against a classifier that
  returns something unexpected.
- **The branch set is fixed at build time.** You cannot add a fourth destination
  based on the input. If the *number* of paths depends on the request, you need
  the orchestrator-worker pattern instead.

### Next Steps
- `6.3_Parallelization.ipynb` — run several chains at once and merge their output
- Compare against the LangGraph original in
  `05_AI_Agent_Fundamentals/4. Workflow_Pattern/2. Routing/`